# KrVoiceAI · Colab 一键出片(LatentSync 数字人 + AI 智能分镜 + 花字编排)

真人驱动视频 → **CosyVoice 克隆声** → **LatentSync 512 唇形同步数字人** → **AI 分镜配商家素材** → **编排成片**(运镜/转场/花字/卡拉OK字幕/BGM)。

**必须先选 A100**:`代码执行程序 → 更改运行时类型 → A100 GPU`(LatentSync 512 需 24G+ 显存,T4/L4 不够)。然后从上到下逐格运行。

In [ ]:
# 1. 确认 GPU —— 必须是 A100。若显示 T4/L4 请换运行时后重跑
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 拉最新代码(本地改动 push 后,重跑本格即 git pull 同步)
%cd /content
import os
if os.path.isdir('KrVoiceAI'):
    %cd KrVoiceAI
    !git pull -q
else:
    !git clone -q https://github.com/cicy-ai/KrVoiceAI.git
    %cd KrVoiceAI
!git log --oneline -1

In [ ]:
# 3. 挂载 Google Drive(取你的驱动视频 / 商家图)。不用 Drive 可跳过本格
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. 指定素材
#   DRIVER    = 你的胸像不说话视频(驱动 LatentSync 口型)
#   VOICE_REF = 你说话的声音样本(克隆参考,10-30秒)—— 自动扫 latentsync/ 里除驱动视频外最新的音频/视频
#   TEXT/TPL/ASSETS/DESC 同前
import os, glob
DRIVER = '/content/drive/MyDrive/latentsync/VID_20260703_141258.mp4'
TEXT   = '大家好,我是老王。我们家新鲜手工水饺,皮薄馅大、用料实在。今天到店,再送你一份小菜!'
TPL    = 'ecommerce'
ASSETS_DIR = '/content/assets'
DESC   = '门店外景,水饺产品特写,老板本人'

# 自动发现声音样本(除 DRIVER 外最新的 音频/视频 文件)
cand = [f for f in glob.glob('/content/drive/MyDrive/latentsync/*')
        if f.lower().endswith(('.m4a','.mp3','.wav','.aac','.amr','.ogg','.mp4','.mov'))
        and os.path.basename(f) != os.path.basename(DRIVER)]
VOICE_REF = max(cand, key=os.path.getmtime) if cand else DRIVER
print('声音样本:', VOICE_REF if cand else '⚠️ 未找到(将用驱动视频音轨,若不说话会报错)')

# 没有商家图时,先用 3 张占位彩块跑通编排
os.makedirs(ASSETS_DIR, exist_ok=True)
if not glob.glob(ASSETS_DIR+'/*'):
    from PIL import Image
    for i,c in enumerate([(46,94,140),(192,96,58),(59,122,87)]):
        Image.new('RGB',(1080,1920),c).save(f'{ASSETS_DIR}/shop{i}.png')
    print('⚠️ 用占位商家图(3张)。真图请上传到', ASSETS_DIR, '再重跑第5格')
print('驱动视频存在:', os.path.exists(DRIVER))
print('latentsync/ 目录:', sorted(os.path.basename(x) for x in glob.glob('/content/drive/MyDrive/latentsync/*')))

In [ ]:
# 5. 一键出片 —— nohup 后台跑,页面刷新/断连都杀不死(首次约 15-25 分钟)
#    克隆声(VOICE_REF) → 数字人 → AI 分镜 → 编排花字成片 → /content/final.mp4;进度看第 6 格
get_ipython().system(f'''cd /content/KrVoiceAI && rm -f /content/produce.done && nohup bash -c 'bash colab/produce.sh "{DRIVER}" "{TEXT}" "{TPL}" "{VOICE_REF}" "{ASSETS_DIR}" "" "{DESC}"; echo EXIT=$? > /content/produce.done' > /content/produce.log 2>&1 &''')
import time; time.sleep(3)
print(open('/content/produce.log').read()[-600:] or '(启动中)')

In [ ]:
# 6. 看进度(反复运行本格;出现「全流程完成」再跑第 7 格)
import os
print(open('/content/produce.log').read()[-2500:] if os.path.exists('/content/produce.log') else '(尚未启动)')
print('\n== 状态:', open('/content/produce.done').read().strip() if os.path.exists('/content/produce.done') else '还在跑…')

In [ ]:
# 7. 预览 + 下载成品
import os
from base64 import b64encode
from IPython.display import HTML
mp4 = '/content/final.mp4'
print('成品:', mp4, os.path.getsize(mp4)//1024, 'KB')
try:
    from google.colab import files; files.download(mp4)
except Exception as e: print('下载跳过:', e)
HTML(f'<video width=340 controls src="data:video/mp4;base64,{b64encode(open(mp4,"rb").read()).decode()}"></video>')

---
## 说明
- **改代码**:本地改 → `git push` → Colab 重跑**第 2 格**(git pull)→ 重跑第 5 格。
- **纯口播档**:第 4 格把 `ASSETS_DIR` 设成空目录/删掉素材,produce.sh 自动走 `finish.sh`(数字人+字幕+运镜,不编排商家图)。
- **换声**:默认抽驱动视频里本人的声做 CosyVoice 克隆参考;想用别人声,改 produce.sh 第4参(声音参考)。